# 🔬 **MorphoBricks | Preprocessing**
___

This notebook prepares an experiment for the other MorphoBricks bricks. It measures the
brightness of every 2D plane of your images (TIFF or OME-Zarr) and saves one **intensity histogram per
plane** in a table, **plane_histograms.csv**. Later bricks use this table to normalise images per
condition, repeat or image without reading the pixels again.

### What the notebook does

- **Checks your experiment folder:** the folder names become the labels of your images
  (condition, biological repeat, technical repeat).
- **Asks you to confirm how the images are organised:** which dimension is depth, channel or
  time. It shows a preview so you can check.
- **Computes the histograms** and writes them, with a record of what was done, to a separate
  output folder.
- **Shows pooled statistics** per condition, repeat or image.

### Quick workflow

1. Run the **Step 0** cell to start.
2. **Step 1:** paste or **Browse…** to your experiment folder, press **Check folder**, read the
   messages and, if the labels are right, press **Yes, the labels are correct**.
3. **Step 2:** check the role of each axis, press **Show preview**, press
   **Yes, the preview looks right** and then **Confirm axis roles**.
4. **Step 3:** press **Compute histograms**. This writes the table the next bricks need.
5. **Step 4 (optional):** press **Calculate statistics** to compare your groups.

### Before you start

- [ ] Your images are **.tif** or **.tiff** files, or OME-Zarr folders (**.ome.zarr**), all with the same pixel type.
- [ ] They are organised in folders like the example below (up to three levels).
- [ ] You know roughly how many slices, channels and time points they have.

> 🛡️ **Your images are never changed.** The results go to a separate folder next to the
> experiment folder (or one you choose), never inside it.

```text
📂 experiment_2026_10                ← select this folder
├── 📂 control                        condition
│   ├── 📂 bio_rep1                   biological repeat
│   │   ├── 📂 tech_rep1              technical repeat
│   │   │   ├── 📄 image_01.tif
│   │   │   └── 📄 image_02.tif
│   │   └── 📂 tech_rep2
│   │       └── 📄 image_01.tif
└── 📂 treated
    └── 📂 bio_rep1                   (no technical repeat folder → "single")
        └── 📄 image_01.tif
```

___
## 🧰 **Step 0: Start the notebook**

Click the cell below and press **Shift + Enter** (or the ▶ button in the toolbar). A green box
appears when everything is ready.

> 💡 **Tip:** the code of each cell is hidden; you only need to run the cells, from top to bottom,
> one after the other.

In [1]:
# @markdown ### Run to start the preprocessing notebook
import subprocess
import sys
from importlib import import_module, invalidate_caches
from importlib.util import find_spec
from pathlib import Path

# Current version of the notebook the user is running
current_version = "0.0.1"
notebook_name = "MorphoBricks_Preprocessing"


def _start_preprocessing_notebook():
    """Prepare this kernel's environment and return the notebook interface."""
    # This standalone cell can run before the package's Python requirement is checked.
    if sys.version_info < (3, 10):  # noqa: UP036
        raise RuntimeError("Select a Python 3.10 or newer notebook kernel, then run Step 0 again.")

    environment = "Google Colab" if "google.colab" in sys.modules else "local Jupyter"
    print(f"Checking the {environment} environment...", flush=True)
    current = Path.cwd().resolve()
    checkout = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "pyproject.toml").is_file()
            and (folder / "src/morphobricks_preprocessing/__init__.py").is_file()
        ),
        None,
    )
    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("morphobricks_preprocessing")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise RuntimeError(
                "This kernel has already loaded another copy of MorphoBricks preprocessing. "
                "Restart the kernel, then run Step 0 to use this checkout."
            )
        # A local source tree works immediately, including in a git worktree.
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        install_target = ["-e", f"{checkout}[notebook]"]
        origin = f"the local checkout at {checkout}"
    elif find_spec("morphobricks_preprocessing") is not None:
        install_target = ["morphobricks-preprocessing[notebook]"]
        origin = "the installed package"
    else:
        install_target = [
            "morphobricks-preprocessing[notebook] @ "
            "git+https://github.com/CellMigrationLab/morphobricks-preprocessing.git"
        ]
        origin = "the GitHub installation"

    required = ("morphobricks_preprocessing", "numpy", "tifffile", "ipywidgets", "matplotlib")
    missing = [name for name in required if find_spec(name) is None]
    if missing:
        print(f"Installing notebook requirements ({', '.join(missing)})...", flush=True)
        try:
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--disable-pip-version-check",
                    *install_target,
                ],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise RuntimeError(
                "Notebook setup could not install its requirements. Check the installation "
                "output above, internet access and write permissions, then rerun Step 0. "
                "Installing from GitHub also needs Git."
            ) from exc
        invalidate_caches()
        missing = [name for name in required if find_spec(name) is None]
        if missing:
            raise RuntimeError(
                f"Installation finished, but this kernel still cannot find {', '.join(missing)}. "
                "Restart the kernel and run Step 0 again."
            )

    print(f"Using {origin}.\nNotebook Python: {sys.executable}", flush=True)
    try:
        interface = import_module("morphobricks_preprocessing.notebook_ui")
    except ImportError as exc:
        raise RuntimeError(
            "The package was found, but the notebook interface could not be loaded. "
            "Read the original error below. If packages were changed in this session, "
            "restart the kernel and run Step 0 again."
        ) from exc
    return interface.PreprocessingNotebook()


notebook = _start_preprocessing_notebook()
notebook.welcome()

Checking the local Jupyter environment...
Using the local checkout at C:\Users\ivanh\Documents\Git\morphobricks-preprocessing\.claude\worktrees\peaceful-hugle-74d2c8.
Notebook Python: c:\Users\ivanh\miniforge3\envs\morpho_pre\python.exe


___
## 📂 **Step 1: Check the experiment folder**

Run the cell below, then:

1. Paste the path of your experiment folder into **Experiment folder**, or click **Browse…** to
   choose it. The **Output folder** box shows where the results will go; leave it empty to use the
   default.
2. Click **Check folder**. Only folder names and image headers are read, so this is quick.
3. Read the summary tiles and the **Messages**. Red messages must be fixed before you can continue;
   amber ones are for information.
4. Check the **Labels from the folder names** table. If every folder has the right labels,
   press the green **Yes, the labels are correct** button in the yellow box under the table.
   It turns green when confirmed; press **Undo** if you change your mind. Step 3 only runs once
   the labels are confirmed.

> 💡 **Tip:** to change a label, rename the folder and click **Check folder** again.
>
> 🩹 **A damaged image?** Click **Exclude these images and check again** to leave it out, or list
> images under **Exclude images (optional)**, one per line.

In [3]:
# @markdown ### Run to show the folder check
notebook.folder_step()

___
## 🧭 **Step 2: Review the axis roles**

A TIFF or OME-Zarr image is a block of numbers with several dimensions. This step tells the brick
what each dimension means, so that every image can be cut into flat 2D planes:

| Role | Meaning | Example |
|---|---|---|
| **X** | width of the image (columns) | 1024 pixels |
| **Y** | height of the image (rows) | 1024 pixels |
| **Z** | depth: focal planes of a z-stack | 25 slices |
| **C** | channel: stains or colours | DAPI, GFP, RFP |
| **T** | time: frames of a time-lapse | 120 time points |

Run the cell below, then:

1. Check the suggested role in each box. The line under each box says what the choice means.
2. Click **Show preview** and move the sliders. Each picture should look like a normal image of your
   sample. Stripes or a smeared picture mean X or Y is on the wrong dimension.
3. If the preview looks right, press **Yes, the preview looks right** in the yellow box under the
   picture. Changing a role afterwards asks you to check the preview again.
4. Press **Confirm axis roles**. The roles are locked until you press **Change axis roles**.

> 💡 **Tip:** if some images lack a dimension the others have (often a single channel), a yellow box
> lists them. Press **Yes, they have a single plane there** if that is the case.

In [4]:
# @markdown ### Run to review the axis roles
notebook.axis_step()

___
## 🚀 **Step 3: Compute the histograms**

Run the cell below. The **Before you run** box lists anything still to do. **What will happen** says
which folder and files will be written and whether an earlier run was found. Then click
**Compute histograms**.

> 💡 **Running again later** only reads images that are new or changed. Rows of deleted or excluded
> images are removed. The recommended settings suit most experiments; untick
> **Use the recommended settings** to change the number of bins, the memory limit, or to recompute
> everything.
>
> 🗂️ **(Optional) Image conversion:** by default the images are read directly and nothing extra is
> saved. You can instead **convert the TIFF images to OME-Zarr** first; the histograms are then computed
> from the converted copies, which are stored in the **ome_zarr** folder of the output. ⚠️ This stores
> the images a second time, so the space they occupy is duplicated; the card shows how much space is
> needed and lets you choose the chunk size, compression and pyramid levels. It needs the zarr package.
>
> 🛡️ Nothing is written unless the whole run succeeds.

In [5]:
# @markdown ### Run to show the settings and the run button
notebook.run_step()

___
## 📊 **Step 4: Calculate the statistics (optional)**

> ℹ️ **This step is optional.** The histogram table written in step 3 is all that the next
> MorphoBricks bricks need. This step only reads it, to let you compare groups and check that the
> brightness looks as expected; it writes no files.

Run the cell below, then:

1. **Histogram table** is filled in with the table from step 3. To look at another experiment,
   paste the path of its plane_histograms.csv or press **Browse…**.
2. Choose how to group the planes (**Condition**, **Biological repeat**, **Technical repeat** or
   **Image**) and the percentiles to report.
3. Press **Calculate statistics**. The table lists pooled statistics per group, and the plot
   compares their intensity distributions.

In [6]:
# @markdown ### Run to show the statistics
notebook.results_step()

___
## ✅ **After running**

- [ ] The output folder contains **plane_histograms.csv** and **run_manifest.json**.
- [ ] Keep both files together; the next MorphoBricks brick reads them.
- [ ] Rerun this notebook whenever you add, change or remove images.
- [ ] If you chose to convert the TIFF images, the output folder also has an **ome_zarr** folder
  with one OME-Zarr copy per TIFF image.

### Common problems

**"The images use different pixel types"**: all images of an experiment must have the same type,
for example 16-bit. Process them separately or convert them.

**"No role is chosen for the axis"**: a dimension with several planes needs Z, C or T in step 2.

**"… needs more working memory than the limit"**: untick *Use the recommended
settings* in step 3 and raise the memory limit, if the computer has enough memory.

**The preview looks like stripes**: X or Y is on the wrong dimension; change the roles in step 2.

More help: the [user guide](../docs/user_guide.md).